In [ ]:
import gc
import torch
import openai
import pandas as pd

from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.representation import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

### Load Data

- Downloaded from Arctic Shift
- r/cybersecurity (global geopolitical security, surveillance, intersection of CS and social issues)
- posts from 2024-01-01 to 2026-04-06 6:20 PM

In [ ]:
posts_df = pd.read_json('r_cybersecurity_posts.jsonl', lines=True)
posts_df

In [ ]:
comments_df = pd.read_json('r_cybersecurity_comments.jsonl', lines=True)
comments_df

#### Data Cleaning
- Remove posts and comments by known reddit bots
- Remove posts and comments that were marked as '\[removed\]' or '\[deleted\]'
- Remove posts with the flairs: 'Career Questions & Discussion', 'Tutorial' as they are not relevant to the social positions we wish to study.\
- Remove posts and comments by '\[deleted\]' authors

In [ ]:
drop_users = {'AutoModerator', 'RemindMeBot', '[deleted]'}
drop_content = {'[removed]', '[deleted]'}
drop_flairs = {'Career Questions & Discussion', 'Tutorial'}
drop_titles = {'[ Removed by moderator ]', '[ Removed by Reddit ]'}

Posts

In [ ]:
clean_posts = posts_df[~posts_df['author'].isin(drop_users)].copy()
clean_posts = clean_posts[~clean_posts['link_flair_text'].isin(drop_flairs)]

clean_posts['selftext'] = clean_posts['selftext'].fillna('')
clean_posts = clean_posts[~clean_posts['selftext'].isin(drop_content)]

clean_posts['title'] = clean_posts['title'].fillna('')
clean_posts = clean_posts[~clean_posts['title'].isin(drop_titles)]

clean_posts

Comments

In [ ]:
clean_comments = comments_df[~comments_df['author'].isin(drop_users)].copy()

clean_comments['body'] = clean_comments['body'].fillna('')
clean_comments = clean_comments[~clean_comments['body'].isin(drop_content)]

# remove all comments from posts with irrelevant flair
valid_post_ids = "t3_" + clean_posts['id'].astype(str)
clean_comments = clean_comments[clean_comments['link_id'].isin(valid_post_ids)]

clean_comments

### 1.1 Aggragate Stats

In [ ]:
postcount = len(clean_posts)
commcount = len(clean_comments)
print('Total Post Count from 2024-01-01 to 2026-04-06:', postcount)
print('Total Comment Count from 2024-01-01 to 2026-04-06:', commcount)

post_authors = set(clean_posts['author'].dropna())
comment_authors = set(clean_comments['author'].dropna())
unique_users = post_authors.union(comment_authors)

print('Total Unique User Count from 2024-01-01 to 2026-04-06:', len(unique_users))

In [ ]:
print('Average Post Score from 2024-01-01 to 2026-04-06:', clean_posts['score'].mean())
print('Median Post Score from 2024-01-01 to 2026-04-06:', clean_posts['score'].median())
print('Maximum Post Score from 2024-01-01 to 2026-04-06:', clean_posts['score'].max())

print()

print('Average Comment Score from 2024-01-01 to 2026-04-06:', clean_comments['score'].mean())
print('Median Comment Score from 2024-01-01 to 2026-04-06:', clean_comments['score'].median())
print('Maximum Comment Score from 2024-01-01 to 2026-04-06:', clean_comments['score'].max())

### Prepare NLP Setup

- Refactor databases to something easier to parse for subsequent tasks
    - Create a merged column with post body + title in posts dataframe
    - Only preserve this column, along with author, creation_utc (date), and score columns
    - Repeat this with column database
    - Concatanate both into a full database
    
- Use BERTopic for assigning topics with its outlined [best practices](https://maartengr.github.io/BERTopic/getting_started/best_practices/best_practices.html)
    - Prevent stochastic behaviour (replicability)
    - Fix number of topics (10)
    - Use MTEB to find most appropriate embeddings for 2026
        - Mihaiii/Ivysaur is better than all-MiniLM-L6-v2 while having the same size
    - Use BERTopic's [integration with LLMs](https://maartengr.github.io/BERTopic/getting_started/representation/llm.html#truncating-documents) to generate topic labels and summaries.
        - Using Google's new gemma4:e2b for **Topic Labelling and Description** based on keywords and documents, locally using Ollama

Refactoring

In [ ]:
p_df = clean_posts[['author', 'created_utc', 'id', 'score']].copy()
p_df['text'] = clean_posts['title'].fillna('') + ": " + clean_posts['selftext'].fillna('')
p_df['type'] = 'post'
p_df['parent_post_id'] = "t3_" + p_df['id'].astype(str) 

c_df = clean_comments[['author', 'created_utc', 'link_id', 'score']].copy()
c_df['text'] = clean_comments['body'].fillna('')
c_df['type'] = 'comment'
c_df['parent_post_id'] = c_df['link_id']

full_df = pd.concat([
    p_df[['author', 'text', 'created_utc', 'type', 'parent_post_id', 'score']],
    c_df[['author', 'text', 'created_utc', 'type', 'parent_post_id', 'score']]
], ignore_index=True)

full_df['date'] = pd.to_datetime(full_df['created_utc'], unit='s').dt.strftime('%Y-%m-%d')

full_df

In [ ]:
# clear out large variables
del posts_df, clean_posts, p_df
del comments_df, clean_comments, c_df
gc.collect()

NLP

In [ ]:
# replicability
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', 
                  random_state=42) # random state is the main setting, the rest are defaults needed
                                   # for BERTopic

# topic count reduction (internal algo)
hdbscan_model = HDBSCAN(min_cluster_size=150, # same idea as UMAP here
                        metric='euclidean', cluster_selection_method='eom', prediction_data=True)

vectorizer_model = CountVectorizer(stop_words="english", min_df=2, ngram_range=(1, 2))
# remove english stop words, look at bigrams for topics, + BERTopic default

client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key='ollama')

prompt_label = """
I have a topic that contains the following documents:
[DOCUMENTS]
The topic is described by the following keywords: [KEYWORDS]

Based on the information above, extract a short topic label (not more than 5 words) in the following format:
topic: <topic label>
"""
label_model = OpenAI(client, model='gemma4:e2b', prompt=prompt_label,
                     generator_kwargs={"reasoning_effort" : "none"})

prompt_summary = """
I have a topic that is described by the following keywords: [KEYWORDS]
In this topic, the following documents are a small but representative subset of all documents in the topic:
[DOCUMENTS]

Based on the information above, please give a description of this topic in the following format:
topic: <description>
"""
summary_model = OpenAI(client, model='gemma4:e2b', prompt=prompt_summary, 
                       generator_kwargs={"reasoning_effort" : "none"})

representation_model = {
   "Label": label_model,
   "Description": summary_model,
}

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

embedding_model = SentenceTransformer("Mihaiii/Ivysaur", device=device)
embeddings = embedding_model.encode(full_df['text'].to_list(), show_progress_bar=True, batch_size=64)

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    representation_model=representation_model,

    nr_topics=21, # one extra is for misc, will be dropped
    verbose=True
)

In [ ]:
topics, _ = topic_model.fit_transform(full_df['text'].to_list(), embeddings)

### 1.2 Topic Identification

In [ ]:
topic_data = topic_model.get_topic_info()
total_convos = postcount + commcount

# formatting
topic_data = topic_data.drop(columns=['Name'])
topic_data = topic_data.drop([0]) # 0th row == -1 : topic for miscallaneous posts and comments as per BERTopic, ignore
topic_data['Share of Total (%)'] = (topic_data['Count']/total_convos)*100
topic_data = topic_data.rename(columns={'Representation':'Top (10) Keywords'})
topic_data = topic_data[['Topic', 'Label', 'Description', 'Count', 'Share of Total (%)', 'Top (10) Keywords', 'Representative_Docs']]


topic_data

### Prepare Remaining Tasks

- Save relevant data
- Clear memory
- Restart notebook after this section

In [ ]:
# this plot will be useful later for validating 1.3
topics_over_time = topic_model.topics_over_time(full_df['text'].to_list(), full_df['date'].to_list())
topic_model.visualize_topics_over_time(topics_over_time, topics=[i for i in range(0, 22)])

In [ ]:
full_df_labels = topic_model.get_document_info(full_df['text'].to_list())
full_df['Topic'] = full_df_labels['Topic']

full_df.to_parquet('cybersec_full_data.pqt')
topic_data.to_parquet('cybersec_topic_data.pqt')

In [ ]:
del topic_model, embeddings
gc.collect()

### 1.3 Trending and Persistent Topics

- Organise data appropriately
    - Key Column: Months
    - Rows: Topics
    - Values: Count of occurences of a given topic for a given month

- Visualise using previous plot: Anything with one or more major spike is defined as a trending topic, the rest are persistent (since they must be persistent enough to be identified by BERTopic)

- Use [Coefficient of Variation](https://en.wikipedia.org/wiki/Coefficient_of_variation) to characterise the above empirically
    - Trending: CV > ??
    - Persistent: CV <= ??

In [ ]:
import pandas as pd

In [ ]:
full_df = pd.read_parquet('cybersec_full_data.pqt')
topic_data = pd.read_parquet('cybersec_topic_data.pqt')

full_df = full_df[full_df['Topic'] > -1] # -1: topic for miscallaneous posts and comments as per BERTopic, ignore

full_df['Month'] = pd.to_datetime(full_df['created_utc'],  unit='s').dt.to_period('M').dt.to_timestamp()
topic_timeline = full_df.groupby(['Month', 'Topic']).size().unstack(fill_value=0)

def classify_by_cv(series, threshold=2.0):
    if series.sum() == 0 or series.mean() == 0:
        return "insignificant"
    cv = series.std() / series.mean()
    if cv > threshold:
        return "trending"
    else:
        return "persistent"

cv_class = topic_timeline.apply(classify_by_cv)

topic_data['Over Time'] = topic_data['Topic'].map(cv_class)
topic_data.to_parquet('cybersec_topic_data.pqt')

topic_data[['Topic', 'Label', 'Over Time']]

### 1.4 Stance and Debate

- Extract dominant position
    - For each topic, get the top 10 most upvoted posts/comments (score column)
    - Pass these to an LLM (gemma4:e2b) with an appropriate prompt to identify the dominant position

- Use zero-shot-classification with valhalla/distilbart-mnli-12-1 to classify 'favourable'/'opposed' labels for each post/comment
    - Use 

- Group by users and take the most frequent stance on each topic to categorise them.

- For each topic, extract 10 top rated posts/comments from each side, and extract arguments using the LLM

In [ ]:
import torch
import openai
import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline

Prompts

In [ ]:
full_df = pd.read_parquet('cybersec_full_data.pqt')
topic_data = pd.read_parquet('cybersec_topic_data.pqt')

prompt_dominance = """
I have a topic labled as '[TOPIC]'. I have some sample documents for the topic:
[DOCUMENTS]

Based on the documents above, extract the short and succinct dominant position with respect to the topic in the following format:
topic: <dominant position>
"""

prompt_argument_fav = """
I have a topic labled as '[TOPIC]'. For this topic, I also have a stance: '[STANCE]'. I have some sample documents that favour the stance:
[DOCUMENTS]

Based on the information above, summarize the key arguments made by the documents in the following format:
stance: <summary>
"""

prompt_argument_opp = """
I have a topic labled as '[TOPIC]'. For this topic, I also have a stance: '[STANCE]'. I have some sample documents that oppose the stance:
[DOCUMENTS]

Based on the information above, summarize the key arguments made by the documents in the following format:
stance: <summary>
"""

client = openai.OpenAI(base_url="http://localhost:11434/v1", api_key='ollama')

def llm_generation(prompt_template, topic_label, documents, stance=None):
    docs_string = "\n".join([f"- {doc}" for doc in documents])
    
    prompt = prompt_template.replace("[TOPIC]", topic_label).replace("[DOCUMENTS]", docs_string)
    if stance:
        prompt = prompt.replace("[STANCE]", stance)
    
    print(prompt)

    response = client.chat.completions.create(
        model="gemma4:e2b",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    
    output = response.choices[0].message.content.strip()
    if ":" in output:
        return output.split(":", 1)[1].strip() 
        # grab everything after ':' and clear up whitespaces on either side
    return output

Dominant Stance

In [ ]:
topic_pos = []
for topic_id in tqdm(topic_data['Topic'].tolist()):

    topic_label = topic_data[topic_data['Topic'] == topic_id]['Label'].values[0][0]
    top_docs = full_df[full_df['Topic'] == topic_id].nlargest(10, 'score')['text'].tolist()

    #print(top_docs[0])

    dom_pos = llm_generation(prompt_dominance, topic_label, top_docs)
    topic_pos.append(dom_pos)

topic_data['Dominant Position'] = topic_pos
topic_data.to_parquet('cybersec_topic_data.pqt')

topic_data

Stance Classification

In [ ]:
topic_data = pd.read_parquet('cybersec_topic_data.pqt')
full_df = pd.read_parquet('cybersec_full_data.pqt')

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

classifier = pipeline("zero-shot-classification", model="valhalla/distilbart-mnli-12-1", device=device)

full_df['Dominant Position Alignment'] = "NA"

threshold = 0.75
for _, row in tqdm(topic_data.iterrows(), total=len(topic_data), desc='Topic'):
    topic_id = row['Topic']
    dom_pos = row['Dominant Position']
    
    mask = (full_df['Topic'] == topic_id) #& (full_df['text'].str.len() >= 200)
    subset_indices = full_df[mask].index
    texts = full_df.loc[subset_indices, 'text'].tolist()
    
    if not texts:
        continue

    topic_stances = []
    batch_size = 64
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i + batch_size]
        results = classifier(
            batch_texts, 
            candidate_labels=["favourable", "opposed"], 
            hypothesis_template=f"The stance toward {dom_pos} is {{}}."
        )
        for res in results:
            top_label = res['labels'][0]
            top_score = res['scores'][0]
            
            if top_score >= threshold:
                topic_stances.append(top_label)
            else:
                topic_stances.append('NA')
    
    full_df.loc[subset_indices, 'Dominant Position Alignment'] = topic_stances

full_df.to_parquet('cybersec_full_data.pqt')
full_df

User Grouping

In [ ]:
topic_data = pd.read_parquet('cybersec_topic_data.pqt')
full_df = pd.read_parquet('cybersec_full_data.pqt')

user_topic_stances = full_df.groupby(['author', 'Topic', 'Dominant Position Alignment']).size().unstack(fill_value=0)

user_topic_stances['Dominant Position Alignment'] = user_topic_stances.idxmax(axis=1)

equal_mask = user_topic_stances['favourable'] == user_topic_stances['opposed']
user_topic_stances.loc[equal_mask, 'Dominant Position Alignment'] = 'NA'

user_topic_stances = user_topic_stances.reset_index()

stance_counts = user_topic_stances[
    user_topic_stances['Dominant Position Alignment'] != 'NA'].groupby(
        ['Topic', 'Dominant Position Alignment']).size().unstack(fill_value=0)

topic_data['Favourable User Count'] = topic_data['Topic'].map(stance_counts['favourable']).fillna(0).astype(int)
topic_data['Opposed User Count'] = topic_data['Topic'].map(stance_counts['opposed']).fillna(0).astype(int)

user_topic_stances.to_parquet('cybersec_user_data.pqt')
topic_data.to_parquet('cybersec_topic_data.pqt')

display(topic_data)
user_topic_stances

Argument Summary

In [ ]:
topic_data = pd.read_parquet('cybersec_topic_data.pqt')
full_df = pd.read_parquet('cybersec_full_data.pqt')

fav_summaries = []
opp_summaries = []

for topic_id in tqdm(topic_data['Topic'].tolist()):
    topic_label = topic_data[topic_data['Topic'] == topic_id]['Label'].values[0][0]
    dom_pos = topic_data[topic_data['Topic'] == topic_id]['Dominant Position'].values[0]
    
    topic_subset = full_df[full_df['Topic'] == topic_id]
    
    supp_docs = topic_subset[topic_subset['Dominant Position Alignment'] == 'favourable'].nlargest(10, 'score')['text'].tolist()
    opp_docs = topic_subset[topic_subset['Dominant Position Alignment'] == 'opposed'].nlargest(10, 'score')['text'].tolist()
    
    if supp_docs:
        supp_arg = llm_generation(prompt_argument_fav, topic_label, supp_docs, stance=f"{dom_pos}")
    else:
        supp_arg = "No significant supporting arguments found."
        
    if opp_docs:
        opp_arg = llm_generation(prompt_argument_opp, topic_label, opp_docs, stance=f"{dom_pos}")
    else:
        opp_arg = "No significant opposing arguments found."
        
    fav_summaries.append(supp_arg)
    opp_summaries.append(opp_arg)

topic_data['Arguments in Favour'] = fav_summaries
topic_data['Arguments in Opposition'] = opp_summaries

topic_data.to_parquet('cybersec_topic_data.pqt')
topic_data